In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
customer_df = spark.read.table("`external-catalog-test`.default.customers_dataset")
display(customer_df)

In [0]:
display(customer_df.groupBy('customer_state').count())

In [0]:
from pyspark.sql import Row

# Create a new row to insert
new_customer = Row(customer_id="9999", customer_unique_id="john_doe_unique_id", customer_zip_code_prefix=90001, customer_city="Los Angeles", customer_state="CA", _rescued_data=None)

# Convert to DataFrame
new_customer_df = spark.createDataFrame([new_customer])

# Append the new row to the existing table
new_customer_df.write.mode("append").insertInto("`external-catalog-test`.default.customers_dataset")

In [0]:
display(new_customer_df)

In [0]:
from pyspark.sql.functions import expr
new_customer_df = new_customer_df.withColumn('status', expr("CASE WHEN customer_state = 'BA' THEN 'RICH' ELSE 'POOR' END"))



In [0]:
spark.sql("""
  DELETE FROM `external-catalog-test`.default.customers_dataset
  WHERE customer_id = '9999'
""")

In [0]:
display(new_customer_df)

In [0]:
spark.sql('select * from `external-catalog-test`.default.customers_dataset')

In [0]:
delta_df_1 = spark.read.format('delta').option('version', 0).table('`external-catalog-test`.default.customers_dataset')


In [0]:
display(delta_df_1)

In [0]:
display(spark.sql("DESCRIBE HISTORY `external-catalog-test`.default.customers_dataset"))

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

w =  Window.partitionBy('customer_zip_code_prefix').orderBy('customer_state')

new_df = delta_df_1.withColumn('state_index', row_number().over(w))
display(new_df)


In [0]:
display(new_df)

In [0]:
delta_df_1.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('`external-catalog-test`.default.customers_dataset')

In [0]:
new_partition= spark.sql("select * from `external-catalog-test`.default.customer_partition")

In [0]:
display(new_partition)